# 02 — Feature Engineering

**Projet**  : ObRail — Détection des sous-dessertes ferroviaires

**Objectif** : Préparer les données GTFS pour la modélisation.

### On normalise, encode et découpe en train/validation/test.

**Input**  : ../data/processed/routes_processed.csv
**Output** : ../data/features/trains_features.csv

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os

RANDOM_STATE = 42
os.makedirs('../data/features', exist_ok=True)

df = pd.read_csv('../data/processed/routes_processed.csv')
print(f"Shape : {df.shape}")
df.head()

### Étape 1 — Sélection des features
On garde uniquement les colonnes utiles pour la modélisation.

On supprime les identifiants texte et les colonnes redondantes.

In [ ]:
cols_drop = ['route_id', 'route_short_name', 'route_long_name', 
             'route_type', 'service_type', 'country', 
             'route_name', 'frequency_category', 'trip_count']

df_model = df.drop(columns=cols_drop)

print(f"Features retenues : {df_model.columns.tolist()}")
print(f"Shape : {df_model.shape}")

### Étape 2 — Normalisation

 On normalise log_trip_count pour que les modèles sensibles

 aux échelles (régression logistique, MLP) ne soient pas biaisés.

In [ ]:
scaler = StandardScaler()
df_model['log_trip_count_scaled'] = scaler.fit_transform(
    df_model[['log_trip_count']]
)
df_model = df_model.drop(columns=['log_trip_count'])

print("✅ log_trip_count normalisé")
print(df_model.describe().round(2))

### Étape 3 — Séparation X et y

In [ ]:
X = df_model.drop(columns=['is_underserved'])
y = df_model['is_underserved']

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"\nClasse balance :\n{y.value_counts(normalize=True).mul(100).round(1)}")

### Étape 4 — Découpage train / validation / test

 60% train, 20% validation, 20% test

**stratify=y** pour conserver la balance des classes dans chaque split

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Train      : {X_train.shape[0]} lignes ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation : {X_val.shape[0]} lignes ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test       : {X_test.shape[0]} lignes ({X_test.shape[0]/len(X)*100:.1f}%)")

### Étape 5 — Sauvegarde

In [ ]:
df_model['is_underserved'] = y
df_model.to_csv('../data/features/trains_features.csv', index=False)

print(f"✅ Sauvegardé → data/features/trains_features.csv")
print(f"Shape finale : {df_model.shape}")
print(f"Colonnes : {df_model.columns.tolist()}")